# System Exploration (System)

Parsing the data and understanding it (System attribute)

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from anomaly_detection.etl.load import load_records, load_system_df

plt.style.use('ggplot')

In [2]:
# Path to the data
project_folder = Path.cwd().parent

output_path = project_folder / "data/system_exploration/system"

evtx_path = project_folder / "data/raw/93_syslog.evtx"

evtx_path

WindowsPath('c:/Users/hp/Documents/School/anomaly-detection/data/raw/93_syslog.evtx')

In [3]:
records = load_records(evtx_path)

len(records)

59779

In [4]:
with open(project_folder / "data/processed/record.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(
        records[0],
        indent=4
    ))

records[0]

{'#attributes': {'xmlns': 'http://schemas.microsoft.com/win/2004/08/events/event'},
 'System': {'Provider': {'Name': 'Service Control Manager',
   'Guid': '{555908d1-a6d7-4695-8e1e-26931d2012f4}',
   'EventSourceName': 'Service Control Manager'},
  'EventID': {'#attributes': {'Qualifiers': 16384}, '#text': 7036},
  'Version': 0,
  'Level': 4,
  'Task': 0,
  'Opcode': 0,
  'Keywords': '0x8080000000000000',
  'TimeCreated': {'SystemTime': '2026-03-13T04:58:47.068108Z'},
  'EventRecordID': 210478,
  'Correlation': None,
  'Execution': {'ProcessID': 932, 'ThreadID': 5368},
  'Channel': 'System',
  'Computer': 'ESANTE.carte.com.tn',
  'Security': None},
 'EventData': {'param1': 'Service de transfert intelligent en arrière-plan',
  'param2': 'en cours d’exécution',
  'Binary': '42004900540053002F0034000000'}}

In [5]:
system_df = load_system_df(evtx_path)

system_df.head()

,EventID,Version,Level,Task,Opcode,Keywords,EventRecordID,Channel,Computer,Qualifiers,Provider_Name,Provider_Guid,Provider_EventSourceName,TimeCreated_SystemTime,Correlation_ActivityID,Execution_ProcessID,Execution_ThreadID,Security_UserID
0,7036,0.0,4,0,0.0,0x8080000000000000,210478,System,ESANTE.carte.com.tn,16384.0,Service Control Manager,{555908d1-a6d7-4695-8e1e-26931d2012f4},Service Control Manager,2026-03-13T04:58:47.068108Z,NaN,932.0,5368.0,NaN
1,7040,0.0,4,0,0.0,0x8080000000000000,210479,System,ESANTE.carte.com.tn,16384.0,Service Control Manager,{555908d1-a6d7-4695-8e1e-26931d2012f4},Service Control Manager,2026-03-13T04:58:47.099352Z,NaN,932.0,5368.0,S-1-5-18
2,7036,0.0,4,0,0.0,0x8080000000000000,210480,System,ESANTE.carte.com.tn,16384.0,Service Control Manager,{555908d1-a6d7-4695-8e1e-26931d2012f4},Service Control Manager,2026-03-13T04:58:47.427480Z,NaN,932.0,9040.0,NaN
3,7036,0.0,4,0,0.0,0x8080000000000000,210481,System,ESANTE.carte.com.tn,16384.0,Service Control Manager,{555908d1-a6d7-4695-8e1e-26931d2012f4},Service Control Manager,2026-03-13T05:00:48.183648Z,NaN,932.0,8788.0,NaN
4,7040,0.0,4,0,0.0,0x8080000000000000,210482,System,ESANTE.carte.com.tn,16384.0,Service Control Manager,{555908d1-a6d7-4695-8e1e-26931d2012f4},Service Control Manager,2026-03-13T05:02:48.194776Z,NaN,932.0,4480.0,S-1-5-18


In [6]:
system_df.columns

Index(['EventID', 'Version', 'Level', 'Task', 'Opcode', 'Keywords',
       'EventRecordID', 'Channel', 'Computer', 'Qualifiers', 'Provider_Name',
       'Provider_Guid', 'Provider_EventSourceName', 'TimeCreated_SystemTime',
       'Correlation_ActivityID', 'Execution_ProcessID', 'Execution_ThreadID',
       'Security_UserID'],
      dtype='str')

# EventID

In [7]:
event_df = pd.DataFrame(system_df['EventID'].value_counts())

event_df.to_csv(output_path / "events.csv")

event_df

,count
EventID,
7036,54164
7040,4637
6013,167
16,152
1801,72
...,...
15007,1
10010,1
36,1


# Constants

In [8]:
system_df.astype(str).nunique()

EventID                        68
Version                         3
Level                           3
Task                           23
Opcode                         10
Keywords                       15
EventRecordID               59779
Channel                         1
Computer                        1
Qualifiers                      5
Provider_Name                  26
Provider_Guid                  24
Provider_EventSourceName        6
TimeCreated_SystemTime      59627
Correlation_ActivityID         13
Execution_ProcessID           192
Execution_ThreadID           2494
Security_UserID                 5
dtype: int64

In [9]:
constants_df = system_df[['Channel', 'Computer']]

constants_df.to_csv(output_path / "constants.csv")

constants_df.value_counts()

Channel  Computer           
System   ESANTE.carte.com.tn    59779
Name: count, dtype: int64

# Others

In [10]:
system_df[['Correlation_ActivityID', 'Execution_ProcessID', 'Execution_ThreadID', 'Security_UserID']].notna().mean()

Correlation_ActivityID    0.000786
Execution_ProcessID       0.996805
Execution_ThreadID        0.996805
Security_UserID           0.089664
dtype: float64

In [11]:
system_df.groupby('Provider_Name')['Execution_ProcessID'].nunique()

Provider_Name
BROWSER                                      0
EventLog                                     0
LsaSrv                                       3
Microsoft-Windows-DHCPv6-Client              7
Microsoft-Windows-DNS-Client                 4
Microsoft-Windows-Dhcp-Client                7
Microsoft-Windows-Directory-Services-SAM     4
Microsoft-Windows-DistributedCOM             1
Microsoft-Windows-FilterManager              1
Microsoft-Windows-GroupPolicy                1
Microsoft-Windows-HttpEvent                  1
Microsoft-Windows-Hyper-V-Netvsc             2
Microsoft-Windows-Kernel-Boot                1
Microsoft-Windows-Kernel-General            78
Microsoft-Windows-Kernel-Power               5
Microsoft-Windows-Kernel-Processor-Power     1
Microsoft-Windows-Ntfs                       1
Microsoft-Windows-TPM-WMI                   70
Microsoft-Windows-Time-Service               6
Microsoft-Windows-WinRM                      1
Microsoft-Windows-WindowsUpdateClient        9